In [3]:
import pandas as pd

df=pd.read_json('liquipedia_data/clean_data/fortnite/players.json')

In [5]:
df[df["tier"]=="medium"]

,pageid,pagename,id,alternateid_list,name,type,status,nationalities,region,birthdate,...,earnings_2019,earnings_2020,earnings_2021,earnings_2022,earnings_2023,earnings_2024,earnings_2025,earnings_2026,fncs_wins,tier
6,10541,1lusha,1lusha,[],NaN,player,Active,[Russia],Europe,2004-07-22,...,2520,4089,48533,26438,13598,12592,16823,18650,1,medium
12,11918,2SNgNl,2SNgNl,[],NaN,player,Retired,[Russia],Europe,2003-06-10,...,1925,8455,17120,10125,0,0,0,0,0,medium
25,14223,3vil,3vil,[],Oskar Żeljazkow,player,Retired,[Poland],Europe,1988-08-24,...,23460,0,0,800,0,0,0,0,0,medium
29,8530,4DRStorm,4DRStorm,[],NaN,player,Retired,[United States],North America,2003-05-23,...,128358,22850,14020,4825,900,0,0,0,0,medium
30,17502,4Turtle,4Turtle,[],NaN,player,Retired,[Australia],Oceania,NaN,...,63750,5354,0,0,500,2500,0,0,0,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5719,19652,Zynox,Zynox,[],NaN,player,Active,[United Kingdom],Europe,NaN,...,0,0,0,1400,5950,25225,15925,25088,0,medium
5720,14924,Zyppan,Zyppan,[Zyppaan],Pontus Eek,player,Retired,[Sweden],Europe,2002-07-17,...,19838,0,0,0,0,0,0,0,0,medium
5721,14280,Zyrofnw,Zyrofnw,[],Hector Gael Garcia Vizcarra,player,Active,[Mexico],North America,2007-07-02,...,0,0,0,5050,6850,4200,47450,1800,0,medium
5723,14548,せこさま,Phazma,[SEKO],NaN,player,Retired,[Japan],Asia,NaN,...,11575,4505,3070,13000,10593,0,0,0,0,medium


In [1]:
import json, re, unicodedata
from collections import Counter, defaultdict

PLAYERS = 'liquipedia_data/clean_data/fortnite/players.json'
LEGACY  = 'src/data/fortnite'          # the Wikipedia import

rows = json.load(open(PLAYERS, encoding='utf-8'))
print(len(rows), 'players')

5726 players


In [2]:
def norm(s):
    """Match key for a handle: fold accents, keep a-z0-9."""
    s = unicodedata.normalize('NFKD', s or '')
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]', '', s.lower())

def base(s):
    """'Speedy (BH)' -> 'speedy'"""
    return norm(re.sub(r'[_ ]*\(.*', '', (s or '').replace('_', ' ')))

def lines(name):
    return open(f'{LEGACY}/{name}', encoding='utf-8').read().splitlines()

tiers = dict(m.groups() for m in
             (re.search(r"id: '([^']+)'.*?tier: '([^']+)'", l) for l in lines('events.ts')) if m)

legacy = {}
for l in lines('players.ts'):
    m = re.search(r"id: '([^']+)', name: '([^']*)'.*?country: '([^']+)'", l)
    if m:
        legacy[m.group(1)] = (m.group(2), m.group(3))

wiki_wins = Counter()
for l in lines('entries.ts'):
    m = re.search(r"eventId: '([^']+)', placement: (\d+), playerIds: \[([^\]]*)\]", l)
    if not m or int(m.group(2)) != 1 or tiers.get(m.group(1)) != 'fncs':
        continue
    for pid in re.findall(r"'([^']+)'", m.group(3)):
        wiki_wins[legacy.get(pid, (pid, None))] += 1

index = defaultdict(list)
for r in rows:
    for k in {norm(r['id']), norm(r['pagename']),
              *(norm(a) for a in (r.get('alternateid_list') or []))}:
        if k:
            index[k].append(r)

CODE_TO_NAT = {
    'US': 'United States', 'CA': 'Canada', 'GB': 'United Kingdom', 'AU': 'Australia',
    'JP': 'Japan', 'BR': 'Brazil', 'FR': 'France', 'DE': 'Germany', 'SA': 'Saudi Arabia',
    'MX': 'Mexico', 'PL': 'Poland', 'RU': 'Russia', 'AT': 'Austria', 'SE': 'Sweden',
    'DK': 'Denmark', 'NL': 'Netherlands', 'NO': 'Norway', 'IE': 'Ireland', 'IT': 'Italy',
    'ES': 'Spain', 'PT': 'Portugal', 'LT': 'Lithuania', 'LV': 'Latvia', 'SI': 'Slovenia',
    'RS': 'Serbia', 'HR': 'Croatia', 'BA': 'Bosnia and Herzegovina', 'UA': 'Ukraine',
    'KR': 'South Korea', 'SG': 'Singapore', 'MY': 'Malaysia', 'IN': 'India',
    'ID': 'Indonesia', 'PK': 'Pakistan', 'AE': 'United Arab Emirates', 'BH': 'Bahrain',
    'KW': 'Kuwait', 'JO': 'Jordan', 'OM': 'Oman', 'SY': 'Syria', 'CL': 'Chile',
    'AR': 'Argentina', 'CU': 'Cuba', 'NZ': 'New Zealand',
}

ALIASES = {
    'Kalgamer': 'Kalgamer710',
    'Kiryache': 'Kiryache32',
    'Speedy (BH)': 'Speedy',
    'Bobi': 'Bobik1ng',        # id is 'BOBY', page is 'Bobik1ng'
    'Buyuriro': 'Buyuriru',
    'Drobban': 'Drobbаn',      # NB: Cyrillic 'а' — see below
    'Kucha': 'Kocha',
    'Mansoor': 'Mansour',
    'Murlox': 'Murloc',
    'Takamura': 'Ruri',
}

def resolve(handle, country=None):
    hits = index.get(norm(handle)) or index.get(base(handle))
    if not hits:
        return None
    if len(hits) > 1 and country:                       # 142 handles are ambiguous
        same = [r for r in hits if CODE_TO_NAT.get(country) in (r.get('nationalities') or [])]
        if same:
            hits = same
    return max(hits, key=lambda r: r.get('earnings') or 0)

fncs, unmatched = {}, []
for (handle, country), n in wiki_wins.items():
    r = resolve(ALIASES.get(handle, handle), country)
    if r is None:
        unmatched.append(handle)
    else:
        fncs[r['pagename']] = max(fncs.get(r['pagename'], 0), n)

for r in rows:
    r['fncs_wins'] = fncs.get(r['pagename'], 0)

print(len(fncs), 'players given a title count')
print('no Liquipedia page:', ', '.join(sorted(unmatched)))

284 players given a title count
no Liquipedia page: 


In [3]:
EASY_PCT, MEDIUM_PCT = 0.02, 0.20     # top 2% easy, next 18% medium, rest hard

def unusable(r):
    """Rows the games must never touch. This is the bit you'll rewrite."""
    return (r.get('status') or '').lower() == 'passed away' or (r.get('earnings') or 0) <= 0

usable = [r for r in rows if not unusable(r)]
usable.sort(key=lambda r: (-(r.get('earnings') or 0), r['id'].lower()))

easy_cut, med_cut = len(usable) * EASY_PCT, len(usable) * MEDIUM_PCT
for r in rows:
    r['tier'] = 'unused'
for i, r in enumerate(usable, start=1):
    r['tier'] = 'easy' if i <= easy_cut else 'medium' if i <= med_cut else 'hard'

print(Counter(r['tier'] for r in rows))

Counter({'hard': 4543, 'medium': 1022, 'easy': 113, 'unused': 48})


In [4]:
# separators=(',', ':') with indent=2 matches the file's existing style
with open(PLAYERS, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2, separators=(',', ':'))

check = json.load(open(PLAYERS, encoding='utf-8'))
assert all(r['tier'] in {'easy', 'medium', 'hard', 'unused'} for r in check)
assert all(isinstance(r['fncs_wins'], int) and r['fncs_wins'] >= 0 for r in check)
print('saved', len(check), 'rows')

saved 5726 rows


In [5]:
import pandas as pd

df=pd.read_json(PLAYERS)

In [6]:
df.head()

,pageid,pagename,id,alternateid_list,name,type,status,nationalities,region,birthdate,...,earnings_2019,earnings_2020,earnings_2021,earnings_2022,earnings_2023,earnings_2024,earnings_2025,earnings_2026,fncs_wins,tier
0,4814,00flour,00flour,[],Ryan Borst,player,Retired,[United States],North America,1996-09-29,...,125,0,0,0,0,0,0,0,0,hard
1,29814,10bit,10bit,[10bittt],NaN,player,Retired,[Canada],North America,NaN,...,6350,0,0,0,0,0,0,0,0,hard
2,32523,10m,10m,[],NaN,player,Active,[Bahrain],Middle East,NaN,...,0,0,0,200,1350,1000,700,100,0,hard
3,19346,13,13,[],NaN,player,Retired,[China],Asia,NaN,...,8125,0,0,0,0,0,0,0,0,hard
4,28922,1Saud,1Saud,[],NaN,player,Active,[Saudi Arabia],Middle East,NaN,...,0,0,0,0,0,700,2530,0,0,hard


In [7]:
df[df["tier"]=="easy"]

,pageid,pagename,id,alternateid_list,name,type,status,nationalities,region,birthdate,...,earnings_2019,earnings_2020,earnings_2021,earnings_2022,earnings_2023,earnings_2024,earnings_2025,earnings_2026,fncs_wins,tier
41,703,72hrs,72hrs,[],Thomas Mulligan,player,Retired,[United States],North America,1994-05-26,...,40625,1133,0,0,0,0,0,0,0,easy
91,8526,Acorn,Acorn,[],Abdullah Akhras,player,Active,[Canada],North America,2004-06-14,...,15000,72326,165488,138043,224790,187150,187725,129600,5,easy
149,8855,Ajerss,Ajerss,[],Aidan Joseph Bernero,player,Active,[United States],North America,2005-11-19,...,10700,28830,113935,75188,168625,62000,158650,69500,2,easy
213,8539,Anas,Anas,[],Anas El-Abd,player,Active,[Denmark],Europe,2002-12-11,...,26900,75066,255379,1242319,29940,27763,1450,850,0,easy
218,8174,Andilex,Andilex,[],Alexandre Christophe,player,Active,[France],Europe,2003-01-31,...,81550,186308,152144,78258,25947,56219,2425,0,1,easy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5397,8163,Wolfiez,Wolfiez,[],Jaden Ashman,player,Retired,[United Kingdom],Europe,2003-12-11,...,1194017,169239,5015,950,0,0,0,0,0,easy
5559,11026,ZAndy,zAndy,[],Andrei Ursu,player,Active,[Romania],Europe,2006-01-17,...,4310,7600,8655,205600,90128,3450,3150,450,0,easy
5585,15053,Zand,Zand,[],Martin Fløjgaard,player,Retired,[Denmark],Europe,NaN,...,341250,0,0,0,0,0,0,0,0,easy
5600,2693,Zayt,Zayt,[],Williams Aubin,player,Retired,[Canada],North America,2000-03-02,...,959525,81003,12377,0,0,0,0,0,1,easy
